In [1]:
import os # get working directory
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl

In [9]:
os.getcwd()

'/workspaces/myfolder/project/sasviya_challenge_2026/notebooks'

## Explaratory Data Analysis (EDA)

### Preliminary assessment on train set

In [3]:
hospital_los_train = pd.read_csv("/workspaces/myfolder/project/sasviya_challenge_2026/data/table_VFL_2026_TRAIN_SET.csv")
hospital_los_train.head(10)

,ENCOUNTER_KEY,PATIENT_NUMBER,DOCTOR,ADMIT_DATE,DISCHARGE_DATE,ICU_DAYS,DEPARTMENT,DISCHARGED_TO,STANDARD_ORDERS_USED,NUM_CHRONIC_COND,...,PROCEDURE_SUBCAT_DESC,PROCEDURE_ICD_CODE,PROCEDURE_LONG_DESC,DX_CODE,DX_GROUP,OPERATION_COUNT,HOSPITAL,ADMIT_MTH,ADMIT_LOS,NUM_VISITS
0,105256104,9921916104,275709,15OCT2011,27OCT2011,9,HEART,"ROUTINE DSCHG, HOME",Y,0,...,OTHER NONOPERATIVE PROCE,99.61,ATRIAL CARDIOVERSION,42833,Congestive heart failure; nonhypertensive [108.],1,Hosp 33,10,12,2
1,105256105,9921916105,275709,15OCT2011,27OCT2011,9,HEART,"ROUTINE DSCHG, HOME",Y,0,...,OTHER NONOPERATIVE PROCE,99.61,ATRIAL CARDIOVERSION,42833,Congestive heart failure; nonhypertensive [108.],0,Hosp 7,10,12,6
2,105256106,9921916106,284130,25NOV2011,27NOV2011,0,HEART,"ROUTINE DSCHG, HOME",Y,2,...,NaN,.,NaN,42823,Congestive heart failure; nonhypertensive [108.],0,Hosp 3,11,2,1
3,105256107,9921916107,284130,25NOV2011,27NOV2011,0,HEART,"ROUTINE DSCHG, HOME",Y,2,...,NaN,.,NaN,42823,Congestive heart failure; nonhypertensive [108.],1,Hosp 21,11,2,5
4,105256108,9921916108,287656,15DEC2011,17DEC2011,0,HEART,"ROUTINE DSCHG, HOME",Y,0,...,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,42833,Congestive heart failure; nonhypertensive [108.],0,Hosp 37,12,2,2
5,105256109,9921916109,287656,15DEC2011,17DEC2011,0,HEART,"ROUTINE DSCHG, HOME",Y,0,...,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,42833,Congestive heart failure; nonhypertensive [108.],2,Hosp 1,12,2,1
6,105256110,9921916110,319038,03JUN2012,13JUN2012,7,HEART,HOME HEALTH AGENCY,Y,1,...,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,42823,Congestive heart failure; nonhypertensive [108.],0,Hosp 39,6,10,3
7,105256111,9921916111,319038,03JUN2012,13JUN2012,7,HEART,HOME HEALTH AGENCY,Y,1,...,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,42823,Congestive heart failure; nonhypertensive [108.],0,Hosp 15,6,10,2
8,105256112,9921916112,235415,04MAY2012,09MAY2012,2,HEART,"ROUTINE DSCHG, HOME",Y,2,...,NaN,.,NaN,42823,Congestive heart failure; nonhypertensive [108.],1,Hosp 37,5,5,4
9,105256113,9921916113,235415,04MAY2012,09MAY2012,2,HEART,"ROUTINE DSCHG, HOME",Y,2,...,NaN,.,NaN,42823,Congestive heart failure; nonhypertensive [108.],1,Hosp 2,5,5,0


In [12]:
# preliminary dataset assessment for training set - shape, dtypes, missingness, cardinality in one table
def dataset_overview(df, name = "dataset"):
    overview = pd.DataFrame({
        'dtype': df.dtypes,
        'null_count': df.isnull().sum(),
        'missing_count': df.isna().sum(),
        'missing_percentage': (df.isnull().sum() / len(df) * 100).round(2),
        'unique': df.nunique()
    })
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns\n")
    return overview.sort_values('missing_percentage', ascending = False)

hospital_los_train_overview = dataset_overview(hospital_los_train, "hospital_los_train")
hospital_los_train_overview

hospital_los_train: 127802 rows, 45 columns



,dtype,null_count,missing_count,missing_percentage,unique
PROCEDURE_LONG_DESC,object,40876,40876,31.98,64
PROCEDURE_SUBCAT_DESC,object,40876,40876,31.98,25
DOCTOR,int64,0,0,0.00,98
PATIENT_NUMBER,int64,0,0,0.00,127802
ENCOUNTER_KEY,int64,0,0,0.00,127802
ICU_DAYS,int64,0,0,0.00,26
DEPARTMENT,object,0,0,0.00,10
DISCHARGED_TO,object,0,0,0.00,11
STANDARD_ORDERS_USED,object,0,0,0.00,2
NUM_CHRONIC_COND,object,0,0,0.00,6


In [14]:
# distinct entries + counts for code/desc column groups
# also checks whether each code maps to exactly one description

column_groups = [
    ('DIAGNOSIS_GROUP', None),
    ('ICD9_TARGET', None),
    ('MS_DRG_CODE', 'MS_DRG_DESC'),
    ('DRG_APR_CODE', 'DRG_APR_DESC'),
    ('DRG_APR_SEVERITY', None),
    ('DIAGNOSIS_SUBCAT_CODE', 'DIAGNOSIS_SUBCAT_DESC'),
    ('DIAGNOSIS_ICD_CODE', 'DX_CODE'),
    ('DIAGNOSIS_LONG_DESC', None),
    ('PROCEDURE_SUBCAT_CODE', 'PROCEDURE_SUBCAT_DESC'),
    ('PROCEDURE_ICD_CODE', 'PROCEDURE_LONG_DESC'),
]

def distinct_counts_and_mapping_check(df, col, paired_col=None):
    print(f"\n{'='*60}\n{col}" + (f"  <->  {paired_col}" if paired_col else "") + f"\n{'='*60}")
    counts = df[col].value_counts(dropna=False).reset_index()
    counts.columns = [col, 'count']
    print(counts.head(15))  # top 15, adjust as needed

    if paired_col and paired_col in df.columns:
        mapping_check = df.groupby(col)[paired_col].nunique()
        inconsistent = mapping_check[mapping_check > 1]
        if len(inconsistent) > 0:
            print(f"\n {len(inconsistent)} value(s) of {col} map to MULTIPLE {paired_col} values:")
            print(inconsistent)
        else:
            print(f"\n Each {col} maps to exactly one {paired_col} value")

for col, paired in column_groups:
    if col in hospital_los_train.columns:
        distinct_counts_and_mapping_check(hospital_los_train, col, paired)


DIAGNOSIS_GROUP
  DIAGNOSIS_GROUP  count
0             CHF  83126
1             AMI  34903
2            COPD   9773

ICD9_TARGET
   ICD9_TARGET   count
0            1  111867
1            0   15935

MS_DRG_CODE  <->  MS_DRG_DESC
    MS_DRG_CODE  count
0           292  34441
1           291  33849
2           293  15090
3           249   5909
4           280   4771
5           191   3592
6           190   3113
7           192   3068
8           281   2957
9           234   2838
10          233   2582
11          282   1768
12          247   1371
13          248   1270
14          246   1049

 Each MS_DRG_CODE maps to exactly one MS_DRG_DESC value

DRG_APR_CODE  <->  DRG_APR_DESC
   DRG_APR_CODE  count
0           194  81819
1           139  30084
2           140   8348
3           137   2027
4           131   1802
5           130    858
6           190    439
7           894    340
8           892    338
9           207    262
10          121    257
11          191    175
12        @@@

In [ ]:
# Hierarchy check — do ID fields nest cleanly inside each othe
id_cols = ['PATIENT_NUMBER', 'DOCTOR', 'DISCH_NURSE_ID', 'HOSPITAL']

print("Unique counts:")
print(hospital_los_train[id_cols].nunique())

def spread_table(df, group_col, target_col):
    # Table: how many <group_col> entities are linked to each count of distinct <target_col> values
    per_entity = df.groupby(group_col)[target_col].nunique()
    table = (
        per_entity.value_counts()
        .reset_index()
    )
    table.columns = [f'num_{target_col}s_linked', f'count_of_{group_col}s']
    table[f'pct_of_{group_col}s'] = (
        table[f'count_of_{group_col}s'] / table[f'count_of_{group_col}s'].sum() * 100
    ).round(1)
    return table.sort_values(f'num_{target_col}s_linked', ascending=False).reset_index(drop=True)

print("DOCTOR -> HOSPITAL spread:")
print(spread_table(hospital_los_train, 'DOCTOR', 'HOSPITAL').to_string(index=False))

print("\nDISCH_NURSE_ID -> HOSPITAL spread:")
print(spread_table(hospital_los_train, 'DISCH_NURSE_ID', 'HOSPITAL').to_string(index=False))

Unique counts:
PATIENT_NUMBER    127802
DOCTOR                98
DISCH_NURSE_ID        93
HOSPITAL              39
dtype: int64
DOCTOR -> HOSPITAL spread:
 num_HOSPITALs_linked  count_of_DOCTORs  pct_of_DOCTORs
                   39                72            73.5
                   38                 2             2.0
                   37                 1             1.0
                   36                 4             4.1
                   35                 6             6.1
                   34                 4             4.1
                   33                 3             3.1
                   32                 1             1.0
                   27                 1             1.0
                   26                 2             2.0
                   25                 1             1.0
                   23                 1             1.0

DISCH_NURSE_ID -> HOSPITAL spread:
 num_HOSPITALs_linked  count_of_DISCH_NURSE_IDs  pct_of_DISCH_NURSE_IDs
         

### Preliminary assessment on test set

In [4]:
hospital_los_test = pd.read_csv("/workspaces/myfolder/project/sasviya_challenge_2026/data/table_VFL_2026_TEST_SET.csv")
hospital_los_test.head(10)

,ENCOUNTER_KEY,PATIENT_NUMBER,DOCTOR,ICU_DAYS,DEPARTMENT,DISCHARGED_TO,STANDARD_ORDERS_USED,NUM_CHRONIC_COND,DISCH_NURSE_ID,ORDER_SET_USED,...,PROCEDURE_SUBCAT_CODE,PROCEDURE_SUBCAT_DESC,PROCEDURE_ICD_CODE,PROCEDURE_LONG_DESC,DX_CODE,DX_GROUP,OPERATION_COUNT,HOSPITAL,ADMIT_MTH,NUM_VISITS
0,105256127,9921916127,220194,0,TRANSPLANT,"ROUTINE DSCHG, HOME",Y,1,10004,1,...,38,"INCISION, EXCISION, AND",38.91,ARTERIAL CATHETERIZATION,4829,Pneumonia (except that caused by TB or STD) [1...,1,Hosp 10,5,2
1,105256137,9921916137,283122,0,ONCOLOGY,"ROUTINE DSCHG, HOME",Y,0,350014,1,...,88,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,4280,Congestive heart failure; nonhypertensive [108.],1,Hosp 35,7,7
2,105256138,9921916138,283122,0,ONCOLOGY,"ROUTINE DSCHG, HOME",Y,0,14006,1,...,88,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,4280,Congestive heart failure; nonhypertensive [108.],0,Hosp 14,7,0
3,105256139,9921916139,262051,0,HEART,"ROUTINE DSCHG, HOME",Y,0,14006,1,...,88,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,4280,Congestive heart failure; nonhypertensive [108.],0,Hosp 14,10,0
4,105256195,9921916195,296744,1,HEART,"ROUTINE DSCHG, HOME",Y,2,19008,1,...,.,NaN,.,NaN,4280,Congestive heart failure; nonhypertensive [108.],1,Hosp 19,9,0
5,105256196,9921916196,319038,5,HEART,HOME HEALTH AGENCY,Y,1,300012,1,...,88,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,42833,Congestive heart failure; nonhypertensive [108.],0,Hosp 30,2,0
6,105256198,9921916198,300214,1,HEART,"ROUTINE DSCHG, HOME",Y,0,15006,1,...,88,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,42833,Congestive heart failure; nonhypertensive [108.],1,Hosp 15,11,1
7,105256219,9921916219,261991,1,GENERAL MED,"ROUTINE DSCHG, HOME",N,0,330013,0,...,96,NONOPERATIVE INTUBATION,96.71,CONTINUOUS INVASIVE MECHANICAL VENTILATION FOR...,486,Pneumonia (except that caused by TB or STD) [1...,0,Hosp 33,1,19
8,105256226,9921916226,262051,1,HEART,HOME HEALTH AGENCY,Y,0,444797,1,...,.,NaN,.,NaN,42823,Congestive heart failure; nonhypertensive [108.],1,Hosp 37,5,0
9,105256228,9921916228,306822,4,HEART,SKILLED NURSING FACIL,N,1,16006,0,...,88,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,42822,Congestive heart failure; nonhypertensive [108.],2,Hosp 16,12,0


In [9]:
# preliminary dataset assessment for test set - shape, dtypes, missingness, cardinality in one table
def dataset_overview(df, name = "dataset"):
    overview = pd.DataFrame({
        'dtype': df.dtypes,
        'null_count': df.isnull().sum(),
        'missing_count': df.isna().sum(),
        'missing_percentage': (df.isnull().sum() / len(df) * 100).round(2),
        'unique': df.nunique()
    })
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns\n")
    return overview.sort_values('missing_percentage', ascending = False)

hospital_los_test_overview = dataset_overview(hospital_los_test, "hospital_los_test")
hospital_los_test_overview

hospital_los_test: 14200 rows, 42 columns



,dtype,null_count,missing_count,missing_percentage,unique
PROCEDURE_SUBCAT_DESC,object,4468,4468,31.46,25
PROCEDURE_LONG_DESC,object,4468,4468,31.46,64
DOCTOR,int64,0,0,0.00,98
ICU_DAYS,int64,0,0,0.00,26
DEPARTMENT,object,0,0,0.00,10
DISCHARGED_TO,object,0,0,0.00,11
STANDARD_ORDERS_USED,object,0,0,0.00,2
NUM_CHRONIC_COND,object,0,0,0.00,6
ENCOUNTER_KEY,int64,0,0,0.00,14200
PATIENT_NUMBER,int64,0,0,0.00,14200
